In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
VOLUME = "volume1"

NS = f"{CATALOG}.{SCHEMA}"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

SHIPMENTS_PATH = f"{BASE}/landing/shipments"
PRODUCTS_PATH = f"{BASE}/landing/product_limits"
DELIVERIES_PATH = f"{BASE}/landing/delivery_events"

RUN_ID = "manual_001"

print("Namespace:", NS)
print("Base:", BASE)
print("Run ID:", RUN_ID)

In [0]:
product_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("min_temperature", DoubleType(), True),
    StructField("max_temperature", DoubleType(), True)
])

In [0]:
products_raw = (
    spark.read
    .option("header", True)
    .schema(product_schema)
    .csv(PRODUCTS_PATH)
)

In [0]:
display(products_raw)

In [0]:
products_bronze = (
    products_raw
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

In [0]:
display(products_bronze)

In [0]:
(
    products_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.bronze_product_limits"
    )
)

In [0]:
display(
    spark.table(
        f"{NS}.bronze_product_limits"
    )
)

In [0]:
shipment_schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", StringType(), True),
    StructField("dispatch_time", TimestampType(), True),
    StructField("expected_arrival", TimestampType(), True)
])

In [0]:
shipments_raw = (
    spark.read
    .option("header", True)
    .schema(shipment_schema)
    .csv(SHIPMENTS_PATH)
)

In [0]:
display(shipments_raw)

In [0]:
shipments_bronze = (
    shipments_raw
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

In [0]:
display(shipments_bronze)

In [0]:
(
    shipments_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.bronze_shipments"
    )
)

In [0]:
display(
    spark.table(
        f"{NS}.bronze_shipments"
    )
)

In [0]:
delivery_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("event_time", TimestampType(), True)
])


In [0]:
deliveries_raw = (
    spark.read
    .schema(delivery_schema)
    .json(DELIVERIES_PATH)
)

display(deliveries_raw)

In [0]:
deliveries_bronze = (
    deliveries_raw
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

In [0]:
(
    deliveries_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.bronze_delivery_events"
    )
)


In [0]:
display(
    spark.table(
        f"{NS}.bronze_delivery_events"
    )
)

In [0]:
SENSOR_LANDING = f"{BASE}/landing/sensor_events"
SENSOR_STAGING = f"{BASE}/staging/sensor_events"

SENSOR_CHECKPOINT = (
    f"{BASE}/checkpoints/bronze_sensor_events"
)

In [0]:
sensor_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("temperature_c", StringType(), True)
])

In [0]:
dbutils.fs.cp(
    f"{SENSOR_STAGING}/batch_001.json",
    f"{SENSOR_LANDING}/batch_001.json"
)

In [0]:
sensor_stream = (
    spark.readStream
    .schema(sensor_schema)
    .option(
        "maxFilesPerTrigger",
        1
    )
    .json(SENSOR_LANDING)
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

In [0]:
display(dbutils.fs.ls(SENSOR_LANDING))

In [0]:
query = (
    sensor_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        SENSOR_CHECKPOINT  #spark stores the progress here
    )
    .trigger(availableNow=True)
    .toTable(
        f"{NS}.bronze_sensor_events"
    )
)

query.awaitTermination()

In [0]:
# %sql

# SELECT *
# FROM training_lab_classic.cold_chain.bronze_sensor_events;

In [0]:
query = (
    sensor_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        SENSOR_CHECKPOINT
    )
    .trigger(availableNow=True)
    .toTable(
        f"{NS}.bronze_sensor_events"
    )
)

query.awaitTermination()

In [0]:
# %sql

# SELECT *
# FROM training_lab_classic.cold_chain.bronze_sensor_events;

In [0]:
# display(
#     dbutils.fs.ls(SENSOR_STAGING)
# )

In [0]:
dbutils.fs.cp(
    f"{SENSOR_STAGING}/batch_002.json",
    f"{SENSOR_LANDING}/batch_002.json"
)

In [0]:
display(
    dbutils.fs.ls(SENSOR_LANDING)
)

In [0]:
sensor_stream = (
    spark.readStream
    .schema(sensor_schema)
    .option(
        "maxFilesPerTrigger",
        1
    )
    .json(SENSOR_LANDING)
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

In [0]:
query = (
    sensor_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        SENSOR_CHECKPOINT
    )
    .trigger(availableNow=True)
    .toTable(
        f"{NS}.bronze_sensor_events"
    )
)

query.awaitTermination()

In [0]:
# %sql
# SELECT *
# FROM training_lab_classic.cold_chain.bronze_sensor_events;

In [0]:
print("STAGING FILES")
display(dbutils.fs.ls(SENSOR_STAGING))

print("LANDING FILES")
display(dbutils.fs.ls(SENSOR_LANDING))

In [0]:
# bronze_count = spark.table(
#     f"{NS}.bronze_sensor_events"
# ).count()

# print("Current Bronze sensor count:", bronze_count)

In [0]:
source = f"{SENSOR_STAGING}/batch_003.json"
destination = f"{SENSOR_LANDING}/batch_003.json"

print("Source:", source)
print("Destination:", destination)

dbutils.fs.cp(
    source,
    destination
)

print("Batch 3 copied successfully")

In [0]:
display(
    dbutils.fs.ls(SENSOR_LANDING)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

sensor_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("temperature_c", StringType(), True)
])

In [0]:
sensor_stream = (
    spark.readStream
    .schema(sensor_schema)
    .option("maxFilesPerTrigger", 1)
    .json(SENSOR_LANDING)
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_processing_time",
        F.current_timestamp()
    )
    .withColumn(
        "_run_id",
        F.lit(RUN_ID)
    )
)

In [0]:
SENSOR_CHECKPOINT = (
    f"{BASE}/checkpoints/bronze_sensor_events"
)

query = (
    sensor_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        SENSOR_CHECKPOINT
    )
    .trigger(availableNow=True)
    .toTable(
        f"{NS}.bronze_sensor_events"
    )
)

query.awaitTermination()

print("Streaming ingestion completed")

In [0]:
# sensor_count = spark.table(
#     f"{NS}.bronze_sensor_events"
# ).count()

# print("Bronze sensor count:", sensor_count)

In [0]:
# batch_3_records = (
#     spark.table(f"{NS}.bronze_sensor_events")
#     .filter(
#         F.col("_source_file").contains("batch_003.json")
#     )
# )

# display(batch_3_records)

# print(
#     "Batch 3 records:",
#     batch_3_records.count()
# )

### verify that bad records survived

In [0]:
bronze_sensor = spark.table(
    f"{NS}.bronze_sensor_events"
)

bad_test_records = bronze_sensor.filter(
    F.col("event_id").isNull()
    |
    (F.col("shipment_id") == "SH999")
    |
    (F.col("temperature_c") == "hot")
)

display(bad_test_records)

In [0]:
display(
    bronze_sensor
    .filter(F.col("event_id") == "E015")
)

In [0]:
print("Streaming checkpoint contents:")

display(
    dbutils.fs.ls(SENSOR_CHECKPOINT)
)

### testing that rerun dosent add same entries

In [0]:
# sensor_stream = (
#     spark.readStream
#     .schema(sensor_schema)
#     .option("maxFilesPerTrigger", 1)
#     .json(SENSOR_LANDING)
#     .withColumn(
#         "_source_file",
#         F.col("_metadata.file_path")
#     )
#     .withColumn(
#         "_processing_time",
#         F.current_timestamp()
#     )
#     .withColumn(
#         "_run_id",
#         F.lit(RUN_ID)
#     )
# )

# query = (
#     sensor_stream.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option(
#         "checkpointLocation",
#         SENSOR_CHECKPOINT
#     )
#     .trigger(availableNow=True)
#     .toTable(
#         f"{NS}.bronze_sensor_events"
#     )
# )

# query.awaitTermination()

# print("Re-run completed")

In [0]:
final_count = spark.table(
    f"{NS}.bronze_sensor_events"
).count()

print("Final count:", final_count)